# Tutorial: eQTL Analysis with JaxQTL and TensorQTL using `cellink`

This tutorial demonstrates how to perform eQTL analysis using external tools JaxQTL and TensorQTL through the `cellink` package. The `cellink` package provides a unified interface to these powerful QTL mapping tools, making it easier to perform comprehensive genetic analyses on single-cell datasets. These tools provide powerful statistical methods for detecting quantitative trait loci (QTLs) in genomic data, with JaxQTL offering fast GPU-accelerated analysis and TensorQTL providing comprehensive cis- and trans-QTL mapping capabilities.

This notebook assumes familiarity with single-cell data processing and basic statistical genetics concepts. The `cellink` package provides convenient wrapper functions that handle data preparation and formatting for these external tools. Install JaxQTL with `pip install 'cellink-tools[jaxqtl]'` (see its [repository](https://github.com/mancusolab/jaxqtl)) and TensorQTL with `pip install 'cellink-tools[tensorqtl]'`. For visualization of QTL calling results, please consider checking out the [Tutorial: Pseudobulk eQTL Analysis with `cellink`](https://cellink-docs.readthedocs.io/en/latest/tutorials/pseudobulk_eqtl.html).

## Environment Setup

We begin by importing necessary libraries and defining key parameters for our analysis. The `cellink` package provides wrapper functions for JaxQTL and TensorQTL that automatically handle data formatting and preparation.

In [1]:
import numpy as np
import scanpy as sc

from cellink.resources import get_dummy_onek1k
from cellink.tl.external import run_jaxqtl, run_tensorqtl

# Analysis parameters
n_gpcs = 20
n_epcs = 15
batch_e_pcs_n_top_genes = 2000
chrom = 22
cis_window = 500_000
cell_type = "CD8 Naive"
celltype_key = "predicted.celltype.l2"
original_donor_col = "donor_id"
maf_threshold = 0.05  # test only variants with a minor allele frequency of at least 5%
min_gene_expr_pct = 0.1  # JaxQTL: test only genes expressed in at least 10% of donors

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Load and Prepare Data

We load the OneK1K dataset, which contains both genotype and single-cell expression data. Its gene annotations (Ensembl coordinates), which define genomic positions and cis-windows, are already included. (This is a subset of the full OneK1K dataset, which can be downloaded, and prepared using `get_onek1k()`)

In [2]:
# Load the dataset
dd = get_dummy_onek1k(verify_checksum=False)
print(f"Dataset shape: {dd.shape}")

dd.G.obsm["gPCs"] = dd.G.obsm["gPCs"][dd.G.obsm["gPCs"].columns[:n_gpcs]]

[2026-11-15 20:30:36,522] INFO:root: ~/cellink_data/dummy_onek1k/dummy_onek1k_dataset.dd.h5 already exists
[2026-11-15 20:30:36,523] WARNING:root: No checksum provided, skipping verification
[2026-11-15 20:30:45,145] INFO:root: Loaded dummy OneK1K dataset: (300, 146939, 125913, 34830)
Dataset shape: (300, 146939, 125913, 34830)



## Data Preprocessing

We filter the dataset to focus on a specific cell type and prepare the data for eQTL analysis.


In [3]:
dd.aggregate(obs=["donor_id", "sex", "age"], func="first", add_to_obs=True)

In [4]:
# Filter to specific cell type
dd = dd[..., dd.C.obs[celltype_key] == cell_type, :].copy()
print(f"After cell type filtering: {dd.shape}")

# Add donor-level metadata
dd.G.obs["donor_sex"] = dd.G.obs["sex"]
dd.G.obs["donor_age"] = dd.G.obs["age"]

# Generate random labels for demonstration (replace with real phenotypes)
dd.G.obs["donor_labels"] = np.random.randint(2, size=len(dd.G.obs))

# Keep the raw counts for JaxQTL's count models and log-normalize X for the Gaussian models (JaxQTL
# gaussian, TensorQTL). Normalizing before the chromosome filter computes the library sizes of X over all
# genes; JaxQTL's count-model offset uses the library sizes of the genes passed to it (chromosome 22 here).
dd.C.layers["counts"] = dd.C.X.copy()
sc.pp.normalize_total(dd.C, target_sum=1e4)
sc.pp.log1p(dd.C)

# Filter to specific chromosome for faster analysis
dd = dd.sel(G_var=dd.G.var.chrom == str(chrom), C_var=dd.C.var.chrom == str(chrom)).copy()
print(f"After chromosome {chrom} filtering: {dd.shape}")

After cell type filtering: (294, 146939, 4859, 34830)
After chromosome 22 filtering: (294, 136776, 4859, 888)


To speed up the computation we keep only the SNPs between 41.9 and 45.6 Mb. This region holds the two strongest CD8 naive T cell cis-eQTLs on chromosome 22 (SMDT1 and FAM118A).

In [5]:
dd = dd[:, (dd.G.var["pos"] > 41_900_000) & (dd.G.var["pos"] < 45_600_000), :, :].copy()
print(f"After SNP filtering: {dd.shape}")

After SNP filtering: (294, 17569, 4859, 888)


## JaxQTL Analysis

JaxQTL is a fast, GPU-accelerated tool for QTL mapping. It supports various statistical models and can handle large-scale genomic data efficiently. `run_jaxqtl` runs the `jaxqtl` subcommands `"cis"`, `"nominal"` and `"trans"` (`mode`); every `jaxqtl` option is a parameter of the same name. The count models (`nb`, `poisson`) get the raw counts summed per donor (`counts_layer="counts"`), with the log library size over all genes (`libsize_key="nCount_RNA"`, since `dd` now holds only chr22) as offset, the `gaussian` model the mean log-normalized expression per donor. Genes expressed in fewer than 10% of donors are skipped (`min_gene_expr_pct`): with only a few non-zero donors the per-variant tests are not valid.

### Running JaxQTL with Default Parameters
The basic `cis` mode performs permutation-based cis-QTL mapping: `pvalue_adj` is a Beta-approximated permutation p-value per gene (corrected for the variants tested for that gene, not across genes). This is the standard approach for identifying cis-eQTLs with appropriate multiple testing correction.

In [6]:
# Basic JaxQTL analysis
results_jaxqtl_basic = run_jaxqtl(
    dd,
    prefix="jaxqtl_basic",
    mode="cis",
    model="nb",  # Negative binomial model
    counts_layer="counts",  # raw counts for the count model
    libsize_key="nCount_RNA",  # library sizes over all genes, not only chr22
    maf=maf_threshold,
    min_gene_expr_pct=min_gene_expr_pct,
    window=cis_window,
    additional_covariates=["gPCs"],
    run=True,
)
results_jaxqtl_basic

[2026-11-15 20:30:56,023] INFO:cellink._core.donordata: Aggregated counts to PB
[2026-11-15 20:30:56,024] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:30:56,540] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:30:56,871] INFO:cellink.io._export: Wrote PLINK 1 binary files jaxqtl_basic.bed/.bim/.fam (294 x 17569)
[2026-11-15 20:30:57,021] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl compute-pcs --pheno jaxqtl_basic_phenotype.bed.gz --covar jaxqtl_basic_donor_features.tsv --num-pcs 2 --normalization library-size --transform log1p --libsize jaxqtl_basic_libsize.tsv --out jaxqtl_basic_donor_features_pcs.tsv


[2026-11-15 20:31:44 - INFO] Reading expression and selecting samples
[2026-11-15 20:31:49 - INFO] Loaded 294 samples and 888 genes after sample selection
[2026-11-15 20:31:49 - INFO] Cohort alignment retained 294 samples; excluded 0 expression samples and 0 covariate samples
[2026-11-15 20:31:49 - INFO] Using library sizes from jaxqtl_basic_libsize.tsv
[2026-11-15 20:31:49 - INFO] Normalizing expression with library-size
[2026-11-15 20:31:50 - INFO] Gene filters retained 504 of 888 genes
[2026-11-15 20:31:57 - INFO] Computing 2 gene expression principal components for 294 samples
[2026-11-15 20:32:02 - INFO] Finished computing 2 gene expression principal components
[2026-11-15 20:32:02 - INFO] ExprPC1 proportion of variance explained: 0.0299614871
[2026-11-15 20:32:02 - INFO] ExprPC2 proportion of variance explained: 0.0123649562
[2026-11-15 20:32:02 - INFO] Appending principal components to aligned covariates
[2026-11-15 20:32:02 - INFO] Writing results.
[2026-11-15 20:32:03 - INFO] 

[2026-11-15 20:32:04,526] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl cis --bfile jaxqtl_basic --pheno jaxqtl_basic_phenotype.bed.gz --covar jaxqtl_basic_donor_features_pcs.tsv --model nb --test score --nperm 1000 --window 500000 --normalize-covar --offset jaxqtl_basic_offset.tsv --min-gene-expr-pct 0.1 --maf 0.05 --out jaxqtl_basic


[2026-11-15 20:32:06 - INFO] Reading genotype, phenotype, and covariate data
[2026-11-15 20:32:07 - INFO] Finished reading and aligning genotype, phenotype, covariate data.
[2026-11-15 20:32:07 - INFO] Starting cis-scan.
[2026-11-15 20:43:15 - INFO] Finished cis-scan. Wrote results to jaxqtl_basic.cis.score.perm.parquet.gz.
[2026-11-15 20:43:15 - INFO] Finished! Thank you!


,phenotype_id,chrom,num_var,snp,a1,a0,pos,tss_distance,af,ma_count,...,beta,se,pvalue,pvalue_adj,adj_method,nb_alpha,negloglikelihood,model_converged,result_valid,failure_reason
0,ENSG00000183864,22,120,22_41924487_C_CA,CA,C,41924487,490992,0.079932,47,...,-0.430231,0.250424,8.579421e-02,6.848263e-01,BETA,1.887847e-02,246.646352,True,True,NaN
1,ENSG00000100412,22,368,22_41980960_G_A,A,G,41980960,533129,0.829932,100,...,-0.151734,0.094404,1.079915e-01,9.284294e-01,BETA,1.000000e-09,392.310658,True,True,NaN
2,ENSG00000100410,22,220,22_41950724_A_G,G,A,41950724,491006,0.102041,60,...,0.275643,0.124216,2.648292e-02,4.720562e-01,BETA,1.000000e-09,355.537863,True,True,NaN
3,ENSG00000100413,22,390,22_41994883_CTG_C,C,CTG,41994883,469083,0.416667,245,...,-0.314345,0.120352,9.004726e-03,3.667374e-01,BETA,1.000000e-09,229.055159,True,True,NaN
4,ENSG00000100417,22,468,22_41965078_C_CT,CT,C,41965078,388177,0.093537,55,...,-0.395051,0.159088,1.301983e-02,2.335141e-01,BETA,1.000000e-09,293.856535,True,True,NaN
5,ENSG00000100418,22,569,22_41967157_A_G,G,A,41967157,369128,0.071429,42,...,-0.484320,0.204980,1.813898e-02,4.592123e-01,BETA,1.000000e-09,298.717891,True,True,NaN
6,ENSG00000196419,22,751,22_42147079_G_GAAA,GAAA,G,42147079,525915,0.170068,100,...,-0.115616,0.052601,2.795016e-02,7.621888e-01,BETA,2.291655e-02,594.209713,True,True,NaN
7,ENSG00000100138,22,803,22_42185242_C_T,T,C,42185242,511308,0.061224,36,...,-0.171170,0.072168,1.770123e-02,6.768897e-01,BETA,3.843895e-03,617.579205,True,True,NaN
8,ENSG00000167077,22,999,22_41988121_A_G,G,A,41988121,288617,0.829932,100,...,-0.397694,0.120045,9.234323e-04,1.037095e-01,BETA,1.000000e-09,332.269067,True,True,NaN
9,ENSG00000100147,22,1075,22_41924487_C_CA,CA,C,41924487,123807,0.079932,47,...,-1.124265,0.419559,7.370258e-03,4.704915e-01,BETA,1.000000e-09,133.877440,True,True,NaN


### Advanced JaxQTL Analysis with Custom Parameters
The `cis` mode with `acat=True` uses the Aggregated Cauchy Association Test to combine p-values across multiple variants, providing a powerful approach for detecting associations when there are multiple causal variants in a region. Its `pvalue_adj` is this gene-level ACAT p-value; it is not permutation-calibrated and not corrected across genes.

In [7]:
# Advanced JaxQTL analysis with more parameters
results_jaxqtl_advanced = run_jaxqtl(
    dd,
    prefix="jaxqtl_advanced",
    mode="cis",
    acat=True,  # gene-level ACAT p-values instead of permutations
    model="gaussian",  # Gaussian model
    window=1000000,  # 1Mb window
    test="wald",  # Wald test
    maf=maf_threshold,
    min_gene_expr_pct=min_gene_expr_pct,
    additional_covariates=["gPCs"],
    expression_pcs=5,  # expression PCs added as covariates (jaxqtl compute-pcs)
    normalize_covar=True,
    verbose=True,
    run=True,
)
results_jaxqtl_advanced

[2026-11-15 20:43:30,100] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 20:43:30,109] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:43:33,839] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:43:34,213] INFO:cellink.io._export: Wrote PLINK 1 binary files jaxqtl_advanced.bed/.bim/.fam (294 x 17569)
[2026-11-15 20:43:34,214] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl compute-pcs --pheno jaxqtl_advanced_phenotype.bed.gz --covar jaxqtl_advanced_donor_features.tsv --num-pcs 5 --normalization library-size --transform log1p --verbose --out jaxqtl_advanced_donor_features_pcs.tsv


[2026-11-15 20:44:14 - INFO] Reading expression and selecting samples
[2026-11-15 20:44:16 - INFO] Loaded 294 samples and 888 genes after sample selection
[2026-11-15 20:44:16 - INFO] Cohort alignment retained 294 samples; excluded 0 expression samples and 0 covariate samples
[2026-11-15 20:44:17 - INFO] Using library sizes computed across all input genes before gene filtering
[2026-11-15 20:44:17 - INFO] Normalizing expression with library-size
[2026-11-15 20:44:17 - INFO] Gene filters retained 504 of 888 genes
[2026-11-15 20:44:18 - INFO] Computing 5 gene expression principal components for 294 samples
[2026-11-15 20:44:22 - INFO] Finished computing 5 gene expression principal components
[2026-11-15 20:44:22 - INFO] ExprPC1 proportion of variance explained: 0.013113318
[2026-11-15 20:44:22 - INFO] ExprPC2 proportion of variance explained: 0.0119646342
[2026-11-15 20:44:22 - INFO] ExprPC3 proportion of variance explained: 0.0116209946
[2026-11-15 20:44:22 - INFO] ExprPC4 proportion of

[2026-11-15 20:44:23,183] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl cis --bfile jaxqtl_advanced --pheno jaxqtl_advanced_phenotype.bed.gz --covar jaxqtl_advanced_donor_features_pcs.tsv --model gaussian --test wald --acat --window 1000000 --normalize-covar --min-gene-expr-pct 0.1 --maf 0.05 --verbose --out jaxqtl_advanced


[2026-11-15 20:44:24 - INFO] Reading genotype, phenotype, and covariate data
[2026-11-15 20:44:25 - INFO] Finished reading and aligning genotype, phenotype, covariate data.
[2026-11-15 20:44:25 - INFO] Starting cis-scan.
[2026-11-15 20:44:25 - WARNING] No cis-SNPs found for ENSG00000283633 over region 22:15601911-17615111. Skipping.
[2026-11-15 20:44:25 - WARNING] No cis-SNPs found for ENSG00000177663 over region 22:16084954-18115694. Skipping.
[2026-11-15 20:44:25 - WARNING] No cis-SNPs found for ENSG00000069998 over region 22:16137511-18165287. Skipping.
[2026-11-15 20:44:25 - WARNING] No cis-SNPs found for ENSG00000093072 over region 22:16178790-18258235. Skipping.
[2026-11-15 20:44:25 - WARNING] No cis-SNPs found for ENSG00000131100 over region 22:16592136-18628749. Skipping.
[2026-11-15 20:44:25 - WARNING] No cis-SNPs found for ENSG00000099968 over region 22:16628855-18730855. Skipping.
[2026-11-15 20:44:25 - WARNING] No cis-SNPs found for ENSG00000015475 over region 22:16734138-1

,phenotype_id,chrom,num_var,snp,a1,a0,pos,tss_distance,af,ma_count,beta,se,pvalue,pvalue_adj,adj_method,negloglikelihood,model_converged,result_valid,failure_reason
0,ENSG00000196236,22,79,22_41928618_T_G,G,T,41928618,1071540,0.197279,116,-0.020553,0.009779,3.653040e-02,4.560988e-01,ACAT,-296.777539,True,True,NaN
1,ENSG00000100387,22,226,22_41948907_A_G,G,A,41948907,997559,0.241497,142,-0.058236,0.023925,1.558694e-02,2.043689e-01,ACAT,-25.061770,True,True,NaN
2,ENSG00000100393,22,788,22_42173118_G_C,C,G,42173118,1080525,0.057823,34,0.035575,0.014143,1.248124e-02,9.066008e-01,ACAT,-346.146874,True,True,NaN
3,ENSG00000100395,22,854,22_41924949_CAAAAA_C,C,CAAAAA,41924949,719666,0.074830,44,0.026187,0.011192,2.003989e-02,4.079041e-01,ACAT,-398.553481,True,True,NaN
4,ENSG00000100401,22,969,22_42182590_T_C,C,T,42182590,937810,0.232993,137,0.016645,0.006488,1.085515e-02,5.521227e-01,ACAT,-401.434489,True,True,NaN
5,ENSG00000100403,22,1142,22_42247439_C_CA,CA,C,42247439,945913,0.316327,186,0.021548,0.007766,5.918915e-03,5.912461e-01,ACAT,-339.109707,True,True,NaN
6,ENSG00000183864,22,1411,22_42386396_A_AC,AC,A,42386396,952901,0.190476,112,0.028254,0.009489,3.175040e-03,7.439547e-01,ACAT,-325.204484,True,True,NaN
7,ENSG00000100412,22,1614,22_42474157_C_A,A,C,42474157,1026326,0.105442,62,0.074699,0.021319,5.380012e-04,8.935034e-01,ACAT,-170.590250,True,True,NaN
8,ENSG00000100410,22,1477,22_42458900_G_T,T,G,42458900,999182,0.069728,41,0.088981,0.022770,1.182894e-04,1.086121e-01,ACAT,-206.072557,True,True,NaN
9,ENSG00000100413,22,1626,22_42526579_C_CT,CT,C,42526579,1000779,0.214286,126,0.027676,0.008652,1.548185e-03,7.911980e-01,ACAT,-355.556457,True,True,NaN


### JaxQTL Trans-QTL Analysis
Trans-QTL analysis identifies associations between variants and distant genes, usually on other chromosomes (here all genes and variants are on chromosome 22, so the scan also covers cis pairs). These typically have smaller effect sizes than cis-QTLs and require larger sample sizes to detect.

In [8]:
# Trans-QTL analysis with JaxQTL
results_jaxqtl_trans = run_jaxqtl(
    dd,
    prefix="jaxqtl_trans",
    mode="trans",
    model="nb",
    counts_layer="counts",
    libsize_key="nCount_RNA",  # library sizes over all genes, not only chr22
    maf=maf_threshold,
    min_gene_expr_pct=min_gene_expr_pct,
    additional_covariates=["gPCs"],
    run=True,
)
results_jaxqtl_trans

[2026-11-15 20:44:36,692] INFO:cellink._core.donordata: Aggregated counts to PB
[2026-11-15 20:44:36,693] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:44:37,352] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:44:37,586] INFO:cellink.io._export: Wrote PLINK 1 binary files jaxqtl_trans.bed/.bim/.fam (294 x 17569)
[2026-11-15 20:44:37,805] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl compute-pcs --pheno jaxqtl_trans_phenotype.bed.gz --covar jaxqtl_trans_donor_features.tsv --num-pcs 2 --normalization library-size --transform log1p --libsize jaxqtl_trans_libsize.tsv --out jaxqtl_trans_donor_features_pcs.tsv


[2026-11-15 20:44:39 - INFO] Reading expression and selecting samples
[2026-11-15 20:44:39 - INFO] Loaded 294 samples and 888 genes after sample selection
[2026-11-15 20:44:39 - INFO] Cohort alignment retained 294 samples; excluded 0 expression samples and 0 covariate samples
[2026-11-15 20:44:39 - INFO] Using library sizes from jaxqtl_trans_libsize.tsv
[2026-11-15 20:44:39 - INFO] Normalizing expression with library-size
[2026-11-15 20:44:39 - INFO] Gene filters retained 504 of 888 genes
[2026-11-15 20:44:39 - INFO] Computing 2 gene expression principal components for 294 samples
[2026-11-15 20:44:41 - INFO] Finished computing 2 gene expression principal components
[2026-11-15 20:44:41 - INFO] ExprPC1 proportion of variance explained: 0.0299614871
[2026-11-15 20:44:41 - INFO] ExprPC2 proportion of variance explained: 0.0123649562
[2026-11-15 20:44:41 - INFO] Appending principal components to aligned covariates
[2026-11-15 20:44:41 - INFO] Writing results.
[2026-11-15 20:44:41 - INFO] 

[2026-11-15 20:44:41,374] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl trans --bfile jaxqtl_trans --pheno jaxqtl_trans_phenotype.bed.gz --covar jaxqtl_trans_donor_features_pcs.tsv --model nb --test score --window 500000 --normalize-covar --offset jaxqtl_trans_offset.tsv --min-gene-expr-pct 0.1 --maf 0.05 --out jaxqtl_trans


[2026-11-15 20:44:42 - INFO] Reading genotype, phenotype, and covariate data
[2026-11-15 20:44:43 - INFO] Finished reading and aligning genotype, phenotype, covariate data.
[2026-11-15 20:45:02 - INFO] Finished! Thank you!


,phenotype,snp,beta,se,pvalue,nb_alpha,model_converged,chrom,pos,a1,a0,af,ma_count
0,ENSG00000283633,22_41900699_T_C,0.524737,0.494921,0.289034,1.000000e-09,True,22,41900699,C,T,0.078231,46
1,ENSG00000283633,22_41901918_G_A,0.524737,0.494921,0.289034,1.000000e-09,True,22,41901918,A,G,0.078231,46
2,ENSG00000283633,22_41901964_T_C,0.524737,0.494921,0.289034,1.000000e-09,True,22,41901964,C,T,0.078231,46
3,ENSG00000283633,22_41902161_C_T,0.525251,0.494968,0.288607,1.000000e-09,True,22,41902161,T,C,0.076531,45
4,ENSG00000283633,22_41902597_C_T,0.524737,0.494921,0.289034,1.000000e-09,True,22,41902597,T,C,0.078231,46
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3502417,ENSG00000079974,22_45599087_T_C,0.026932,0.092205,0.770223,1.000000e-09,True,22,45599087,C,T,0.658163,201
3502418,ENSG00000079974,22_45599476_T_C,-0.133552,0.128831,0.299905,1.000000e-09,True,22,45599476,C,T,0.171769,101
3502419,ENSG00000079974,22_45599746_C_T,-0.099005,0.180113,0.582537,1.000000e-09,True,22,45599746,T,C,0.088435,52
3502420,ENSG00000079974,22_45599912_C_T,-0.133552,0.128831,0.299905,1.000000e-09,True,22,45599912,T,C,0.171769,101


## TensorQTL Analysis

TensorQTL provides comprehensive QTL mapping capabilities with support for various analysis modes and statistical approaches. TensorQTL can be run in various modes: `["cis_nominal", "cis_independent", "cis", "trans", "cis_susie", "trans_susie"]`.

TensorQTL models continuous expression: `run_tensorqtl` averages the log-normalized expression per donor.

### Basic Cis-QTL Analysis with TensorQTL

The basic `cis` mode in TensorQTL performs permutation-based cis-QTL mapping, similar to JaxQTL but with PyTorch-based GPU acceleration.

In [9]:
# Basic cis-QTL mapping
results_tensorqtl_cis = run_tensorqtl(
    dd,
    prefix="tensorqtl_cis",
    mode="cis",
    window=cis_window,
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    permutations=10000,
    run=True,
)
results_tensorqtl_cis

[2026-11-15 20:45:04,128] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 20:45:04,129] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:45:04,287] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 20:45:21,146] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:45:21,341] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_cis.pgen/.pvar/.psam (294 x 17569)


site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 20:50:28,095] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 20:50:28,095] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 20:50:28,345] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:50:28,345] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:50:28,374] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 20:50:28,656] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 20:50:28,889] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 20:50:29] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_cis_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_cis_donor_featur

,phenotype_id,num_var,beta_shape1,beta_shape2,true_df,pval_true_df,variant_id,start_distance,end_distance,ma_samples,ma_count,af,pval_nominal,slope,slope_se,pval_perm,pval_beta,qval,pval_nominal_threshold
0,ENSG00000273424,75,0.908110,4.77981,190.609,0.107026,22_41924487_C_CA,493552,493112,46,47,0.079932,0.054991,0.002189,0.001136,0.464654,0.461630,0.696687,0.001986
1,ENSG00000183864,120,0.941163,7.73796,245.232,0.280777,22_41905708_T_C,472213,458907,75,79,0.134354,0.257702,0.012672,0.011172,0.928407,0.929375,0.716208,0.001458
2,ENSG00000100412,368,0.917726,14.73080,227.481,0.025051,22_41992071_G_C,544240,462798,110,120,0.795918,0.014627,-0.044640,0.018166,0.346365,0.351453,0.696687,0.000675
3,ENSG00000100410,220,0.938151,10.97770,229.835,0.001969,22_41921891_CA_C,462173,453199,45,46,0.078231,0.000794,0.072525,0.021372,0.031597,0.027752,0.370646,0.001011
4,ENSG00000100413,390,0.911718,15.83510,229.785,0.065835,22_42008096_CCTTTCCTTTCCTTTCCTTTCCTTTG_C,482296,463490,44,47,0.079932,0.046114,-0.023778,0.011867,0.685331,0.695475,0.696687,0.000608
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,ENSG00000182257,176,0.826408,11.72750,222.286,0.099721,22_45569642_AAAGAAGAAGAAG_A,-479837,-484502,42,44,0.074830,0.069575,0.002531,0.001389,0.756324,0.770843,0.696687,0.000492
70,ENSG00000273289,176,0.731048,10.19490,220.780,0.073602,22_45574777_C_CT,-478317,-478783,42,44,0.074830,0.047828,0.001396,0.000702,0.627037,0.669047,0.696687,0.000280
71,ENSG00000241990,175,0.928557,15.72180,239.984,0.030026,22_45572077_C_T,-481629,-485133,197,251,0.573129,0.021352,-0.022638,0.009778,0.404960,0.416033,0.696687,0.000671
72,ENSG00000197182,175,0.852687,13.14590,232.687,0.168891,22_45596269_C_T,-457601,-517659,170,220,0.625850,0.138286,-0.006457,0.004344,0.941306,0.932680,0.716208,0.000519


### Nominal Cis-QTL Analysis
Nominal mode tests all variant-gene pairs within the specified window and returns comprehensive association statistics. This mode can also output top associations per gene, and significant pairs at the gene-level FDR when the output of a `cis` run is passed as `cis_output`.

In [10]:
# Nominal cis-QTL analysis (all variant-gene pairs)
results_tensorqtl_nominal = run_tensorqtl(
    dd,
    prefix="tensorqtl_nominal",
    mode="cis_nominal",
    window=cis_window,
    pval_threshold=1e-5,
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    batch_size=20000,
    run=True,
)

# Results contain multiple outputs for nominal mode
cis_qtl_pairs, cis_qtl_signif_pairs, cis_qtl_top_assoc = results_tensorqtl_nominal
print(f"Total variant-gene pairs tested: {len(cis_qtl_pairs)}")
print(f"Pairs with p < 1e-5: {(cis_qtl_pairs['pval_nominal'] < 1e-5).sum()}")
cis_qtl_pairs

[2026-11-15 20:51:10,141] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 20:51:10,142] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:51:10,242] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 20:51:10,840] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:51:10,868] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_nominal.pgen/.pvar/.psam (294 x 17569)


site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 20:51:21,044] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 20:51:21,044] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 20:51:21,291] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:51:21,291] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:51:21,315] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 20:51:21,667] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 20:51:21,871] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 20:51:21] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_nominal_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_nominal_dono

,phenotype_id,variant_id,start_distance,end_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se
0,ENSG00000273424,22_41900699_T_C,469764,469324,0.078231,45,46,0.586136,-0.000618,0.001133
1,ENSG00000273424,22_41901918_G_A,470983,470543,0.078231,45,46,0.586136,-0.000618,0.001133
2,ENSG00000273424,22_41901964_T_C,471029,470589,0.078231,45,46,0.586136,-0.000618,0.001133
3,ENSG00000273424,22_41902161_C_T,471226,470786,0.076531,44,45,0.594319,-0.000610,0.001143
4,ENSG00000273424,22_41902597_C_T,471662,471222,0.078231,45,46,0.586136,-0.000618,0.001133
...,...,...,...,...,...,...,...,...,...,...
169425,ENSG00000231010,22_45599087_T_C,-456654,-459073,0.658163,161,201,0.222720,0.000370,0.000303
169426,ENSG00000231010,22_45599476_T_C,-456265,-458684,0.171769,89,101,0.248727,0.000452,0.000391
169427,ENSG00000231010,22_45599746_C_T,-455995,-458414,0.088435,48,52,0.589274,-0.000288,0.000534
169428,ENSG00000231010,22_45599912_C_T,-455829,-458248,0.171769,89,101,0.248727,0.000452,0.000391


### Cis-Independent QTL Analysis
The `cis_independent` mode identifies multiple independent association signals for each gene by performing conditional analysis. This is crucial for understanding the genetic architecture when multiple causal variants affect the same gene.

In [11]:
# First run basic cis mode to get top associations
results_tensorqtl_cis = run_tensorqtl(
    dd,
    prefix="tensorqtl_cis",
    mode="cis",
    window=cis_window,
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    permutations=10000,
    run=True,
)

# Now identify independent signals conditioning on the top hits
results_tensorqtl_independent = run_tensorqtl(
    dd,
    prefix="tensorqtl_independent",
    mode="cis_independent",
    window=cis_window,
    cis_output="tensorqtl_cis.cis_qtl.txt.gz",  # Use output from cis mode
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    run=True,
)

print(f"Independent signals identified: {len(results_tensorqtl_independent)}")
results_tensorqtl_independent

[2026-11-15 20:51:25,920] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 20:51:25,920] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:51:26,017] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 20:51:28,564] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:51:28,570] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_cis.pgen/.pvar/.psam (294 x 17569)


site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 20:51:38,644] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 20:51:38,644] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 20:51:38,893] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:51:38,893] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:51:38,921] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 20:51:39,190] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 20:51:39,395] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 20:51:39] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_cis_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_cis_donor_featur

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 20:52:05,667] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 20:52:05,667] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 20:52:05,920] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:52:05,920] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:52:05,948] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 20:52:06,230] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 20:52:06,458] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 20:52:06] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_independent_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_independ

,phenotype_id,num_var,beta_shape1,beta_shape2,true_df,pval_true_df,variant_id,start_distance,end_distance,ma_samples,ma_count,af,pval_nominal,slope,slope_se,pval_perm,pval_beta,rank
0,ENSG00000183172,1741,1.033390,74.3732,217.370,1.877640e-17,22_42120602_TAA_T,40910,36318,140,164,0.278912,2.627020e-21,-0.356368,0.034488,0.000100,4.393210e-16,1
1,ENSG00000270083,2361,0.708277,16.4785,117.486,7.720600e-12,22_42065183_T_C,-24448,-24845,28,31,0.052721,3.486190e-25,0.245482,0.021333,0.001300,1.070110e-07,1
2,ENSG00000270083,2361,0.718069,17.1367,119.690,9.749260e-10,22_42116188_C_T,26557,26160,29,31,0.052721,4.499270e-20,-0.218861,0.021990,0.006299,2.837060e-06,2
3,ENSG00000100376,3147,0.989898,195.8450,213.779,8.379580e-23,22_45393863_T_G,84894,51908,76,82,0.139456,2.202940e-28,0.302726,0.024338,0.000100,2.611200e-20,1


### Trans-QTL Analysis with TensorQTL
Trans-QTL mapping with TensorQTL tests associations between variants and distant genes. Due to the large number of tests, stricter p-value thresholds are typically required.

In [12]:
# Trans-QTL mapping
results_tensorqtl_trans = run_tensorqtl(
    dd,
    prefix="tensorqtl_trans",
    mode="trans",
    pval_threshold=1e-6,  # Stricter threshold for trans
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    batch_size=10000,  # Smaller batches for trans analysis
    return_r2=True,  # Include R² statistics
    run=True,
)
results_tensorqtl_trans

[2026-11-15 20:52:09,080] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 20:52:09,081] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:52:09,183] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 20:52:09,834] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:52:09,855] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_trans.pgen/.pvar/.psam (294 x 17569)


site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 20:52:20,584] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 20:52:20,584] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 20:52:20,835] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:52:20,835] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:52:20,864] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 20:52:21,143] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 20:52:21,367] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 20:52:21] Running TensorQTL v1.0.10: trans-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_trans_phenotype.bed.gz)
  * reading covariates (tensorqtl_trans_donor_features.tsv)
  * loading genotypes
  * p-value threshold: 1

,variant_id,phenotype_id,pval,b,b_se,r2,af
0,22_41900699_T_C,ENSG00000211660,3.474633e-07,0.008238,0.001576,0.091852,0.078231
2,22_41901918_G_A,ENSG00000211660,3.474633e-07,0.008238,0.001576,0.091852,0.078231
4,22_41901964_T_C,ENSG00000211660,3.474633e-07,0.008238,0.001576,0.091852,0.078231
6,22_41902161_C_T,ENSG00000211660,3.584816e-07,0.008302,0.001591,0.091649,0.076531
8,22_41902597_C_T,ENSG00000211660,3.474633e-07,0.008238,0.001576,0.091852,0.078231
...,...,...,...,...,...,...,...
1199,22_45537816_C_T,ENSG00000100079,4.276185e-07,0.029300,0.005653,0.090501,0.088435
1200,22_45547561_A_T,ENSG00000167037,5.076265e-07,0.090687,0.017616,0.089384,0.078231
1201,22_45574777_C_CT,ENSG00000211666,2.475879e-07,0.006328,0.001195,0.094054,0.074830
1202,22_45574777_C_CT,ENSG00000128311,6.230961e-07,0.015881,0.003111,0.088047,0.074830


### SuSiE Finemapping with TensorQTL
`SuSiE` (Sum of Single Effects) is a powerful finemapping method that identifies credible sets of causal variants for each independent association signal. It accounts for LD structure and provides posterior probabilities for each variant being causal.
#### Cis-SuSiE Finemapping
Cis-SuSiE performs finemapping on the top cis-QTL signals to identify credible sets of potentially causal variants. This requires the output from a previous cis-QTL run.

In [13]:
# First, run standard cis-QTL analysis if not already done
results_tensorqtl_cis = run_tensorqtl(
    dd,
    prefix="tensorqtl_cis_for_susie",
    mode="cis",
    window=cis_window,
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    permutations=10000,
    run=True,
)

# Perform SuSiE finemapping on cis-QTL signals
results_tensorqtl_cis_susie = run_tensorqtl(
    dd,
    prefix="tensorqtl_cis_susie",
    mode="cis_susie",
    window=cis_window,
    cis_output="tensorqtl_cis_for_susie.cis_qtl.txt.gz",  # Input from cis mode
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    max_effects=10,  # Maximum number of causal effects per gene
    run=True,
)

# Results contain SuSiE model objects and summary statistics
susie_models, susie_summary = results_tensorqtl_cis_susie

print(f"Genes with finemapping results: {len(susie_models)}")
print("\nSummary statistics columns:")
print(susie_summary.columns.tolist())
print(f"\nCredible sets identified: {susie_summary.groupby(['phenotype_id', 'cs_id']).ngroups}")

# Examine credible sets for a specific gene
if len(susie_summary) > 0:
    example_gene = susie_summary["phenotype_id"].iloc[0]
    gene_credible_sets = susie_summary[susie_summary["phenotype_id"] == example_gene]
    print(f"\nCredible sets for {example_gene}:")
    print(gene_credible_sets)

[2026-11-15 20:52:23,827] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 20:52:23,827] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:52:23,937] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 20:52:26,725] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:52:26,769] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_cis_for_susie.pgen/.pvar/.psam (294 x 17569)


site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 20:52:37,435] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 20:52:37,436] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 20:52:37,693] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:52:37,693] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:52:37,716] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 20:52:38,000] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 20:52:38,229] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 20:52:38] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_cis_for_susie_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_cis_fo

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 20:53:05,770] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 20:53:05,770] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 20:53:06,039] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:53:06,039] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 20:53:06,068] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 20:53:06,356] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 20:53:06,595] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 20:53:06] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_cis_susie_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_cis_susie_

## Advanced Usage: Dry Run and Command Generation

Both JaxQTL and TensorQTL support generating commands without execution, which is useful for debugging or running on compute clusters. This is controlled by the `run` argument.

### JaxQTL Command Generation

In [14]:
# Generate JaxQTL command without running
jaxqtl_command = run_jaxqtl(
    dd,
    prefix="jaxqtl_cluster",
    mode="cis",
    additional_covariates=["gPCs"],
    counts_layer="counts",  # dd.C.X is log-normalized by now
    libsize_key="nCount_RNA",  # library sizes over all genes, not only chr22
    maf=maf_threshold,
    min_gene_expr_pct=min_gene_expr_pct,
    run=False,  # Don't execute, just return command
)

print("JaxQTL command:")
print(jaxqtl_command)

[2026-11-15 20:53:11,986] INFO:cellink._core.donordata: Aggregated counts to PB
[2026-11-15 20:53:11,987] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:53:12,512] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:53:12,779] INFO:cellink.io._export: Wrote PLINK 1 binary files jaxqtl_cluster.bed/.bim/.fam (294 x 17569)
JaxQTL command:
jaxqtl compute-pcs --pheno jaxqtl_cluster_phenotype.bed.gz --covar jaxqtl_cluster_donor_features.tsv --num-pcs 2 --normalization library-size --transform log1p --libsize jaxqtl_cluster_libsize.tsv --out jaxqtl_cluster_donor_features_pcs.tsv && jaxqtl cis --bfile jaxqtl_cluster --pheno jaxqtl_cluster_phenotype.bed.gz --covar jaxqtl_cluster_donor_features_pcs.tsv --model nb --test score --nperm 1000 --window 500000 --normalize-covar --offset jaxqtl_cluster_offset

### TensorQTL Command Generation

In [15]:
# Generate TensorQTL command without running
tensorqtl_command = run_tensorqtl(
    dd,
    prefix="tensorqtl_cluster",
    mode="cis",
    window=cis_window,
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    run=False,  # Don't execute, just return command
)

print("\nTensorQTL command:")
print(tensorqtl_command)

[2026-11-15 20:53:12,842] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 20:53:12,843] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:53:12,945] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 20:53:13,564] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:53:13,622] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_cluster.pgen/.pvar/.psam (294 x 17569)

TensorQTL command:
python -m cellink.tl.external._tensorqtl_main tensorqtl_cluster tensorqtl_cluster_phenotype.bed.gz tensorqtl_cluster --covariates tensorqtl_cluster_donor_features.tsv --mode cis --permutations 10000 --window 500000 --pval_threshold 1e-05 --maf_threshold 0.05 --maf_threshold_interaction 0.05 --batch_size 20000 --warn_monomorphic --max_effects 10 --fdr 0.05


#### Saving Commands to Files
You can also save the generated commands to files for batch submission to compute clusters:

In [16]:
# Save JaxQTL command to file
jaxqtl_command = run_jaxqtl(
    dd,
    prefix="jaxqtl_cluster",
    mode="cis",
    acat=True,
    model="nb",
    window=cis_window,
    additional_covariates=["gPCs"],
    counts_layer="counts",  # dd.C.X is log-normalized by now
    libsize_key="nCount_RNA",  # library sizes over all genes, not only chr22
    maf=maf_threshold,
    min_gene_expr_pct=min_gene_expr_pct,
    run=False,
    save_cmd_file="jaxqtl_job.sh",
)

[2026-11-15 20:53:13,641] INFO:cellink._core.donordata: Aggregated counts to PB
[2026-11-15 20:53:13,641] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:53:14,081] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:53:14,321] INFO:cellink.io._export: Wrote PLINK 1 binary files jaxqtl_cluster.bed/.bim/.fam (294 x 17569)


In [17]:
# Save TensorQTL command to file
tensorqtl_command = run_tensorqtl(
    dd,
    prefix="tensorqtl_cluster",
    mode="cis_susie",
    cis_output="previous_cis_results.txt.gz",
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    run=False,
    save_cmd_file="tensorqtl_susie_job.sh",
)

[2026-11-15 20:53:14,365] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 20:53:14,366] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 20:53:14,958] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 20:53:15,491] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 20:53:15,497] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_cluster.pgen/.pvar/.psam (294 x 17569)
